# Task 3: Data Visualization

## Objective

Provide the visual storytelling layer for Level 3:

1. Distribution of `Aggregate rating`
2. Comparison of average ratings across cuisines (primary) and major cities (context)
3. Relationships between selected features and the target
4. Visual recap of Task 1 (model comparison) and Task 2 (cuisine popularity) where those outputs exist

Charts use labeled axes, documented aggregation, and top-N subsets when cardinality is high. No 3D or decorative plots.


## Dataset / Input

- Engineered dataset: `../data/engineered_dataset.csv`
- Optional Task 1 table: `../Task-1_Predictive_Modeling/outputs/tables/model_comparison.csv`
- Optional Task 2 table: `../Task-2_Customer_Preference_Analysis/outputs/tables/cuisine_stats_all.csv`


## Imports


In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.dpi"] = 120

BASE_DIR = Path.cwd()
DATA_PATH = (BASE_DIR / "../data/engineered_dataset.csv").resolve()
FIG_DIR = BASE_DIR / "outputs" / "figures"
TAB_DIR = BASE_DIR / "outputs" / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

T1_METRICS = (BASE_DIR / "../Task-1_Predictive_Modeling/outputs/tables/model_comparison.csv").resolve()
T2_STATS = (BASE_DIR / "../Task-2_Customer_Preference_Analysis/outputs/tables/cuisine_stats_all.csv").resolve()
MIN_RESTAURANTS = 30
TOP_N = 15

print("Working directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)


Working directory: C:\Users\adity\Documents\Cognifyz\Level-3\Task-3_Data_Visualization
Dataset path: C:\Users\adity\Documents\Cognifyz\Level-3\data\engineered_dataset.csv


## Load Dataset


In [2]:
df = pd.read_csv(DATA_PATH)
print("Rows:", df.shape[0], "Columns:", df.shape[1])
print("Target missing:", int(df["Aggregate rating"].isna().sum()))
df.head()


Rows: 9551 Columns: 26
Target missing: 0


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Price range,Aggregate rating,Rating color,Rating text,Votes,Restaurant_Name_Length,Address_Length,Has_Table_Booking,Has_Online_Delivery,Cuisine_Count
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,3,4.8,Dark Green,Excellent,314,16,71,1,0,3
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,3,4.5,Dark Green,Excellent,591,16,67,1,0,1
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,4,4.4,Green,Very Good,270,22,56,1,0,4
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,4,4.9,Dark Green,Excellent,365,4,70,0,0,2
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,4,4.8,Dark Green,Excellent,229,11,64,1,0,2


## Data Validation


In [3]:
print(df[["Aggregate rating", "Votes", "Price range", "City", "Cuisines"]].describe(include="all").transpose())
print()
print("City counts (head):")
print(df["City"].value_counts().head(10).to_string())


                   count unique           top  freq        mean         std  \
Aggregate rating  9551.0    NaN           NaN   NaN     2.66637    1.516378   
Votes             9551.0    NaN           NaN   NaN  156.909748  430.169145   
Price range       9551.0    NaN           NaN   NaN    1.804837    0.905609   
City                9551    141     New Delhi  5473         NaN         NaN   
Cuisines            9542   1825  North Indian   936         NaN         NaN   

                  min  25%   50%    75%      max  
Aggregate rating  0.0  2.5   3.2    3.7      4.9  
Votes             0.0  5.0  31.0  131.0  10934.0  
Price range       1.0  1.0   2.0    2.0      4.0  
City              NaN  NaN   NaN    NaN      NaN  
Cuisines          NaN  NaN   NaN    NaN      NaN  

City counts (head):
City
New Delhi       5473
Gurgaon         1118
Noida           1080
Faridabad        251
Ghaziabad         25
Ahmedabad         21
Amritsar          21
Bhubaneshwar      21
Guwahati          21
Luck

## Visualization Group 1 — Rating Distribution

The histogram shows central tendency, spread, the unrated spike at 0.0, and the concentration of scores between roughly 2.5 and 4.0.


In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df["Aggregate rating"], bins=np.arange(-0.05, 5.05, 0.1), ax=ax, color="steelblue", edgecolor="white")
ax.axvline(df["Aggregate rating"].mean(), color="black", linestyle="--", linewidth=1, label=f"Mean = {df['Aggregate rating'].mean():.2f}")
ax.axvline(df["Aggregate rating"].median(), color="firebrick", linestyle=":", linewidth=1.5, label=f"Median = {df['Aggregate rating'].median():.2f}")
ax.set_title("Distribution of Aggregate Rating")
ax.set_xlabel("Aggregate rating")
ax.set_ylabel("Number of restaurants")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "rating_distribution_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

print("Share rated 0.0:", round((df["Aggregate rating"] == 0).mean(), 4))
print("Skewness:", round(float(df["Aggregate rating"].skew()), 3))


Share rated 0.0: 0.2249
Skewness: -0.954


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\869241332.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
order_text = ["Not rated", "Poor", "Average", "Good", "Very Good", "Excellent"]
present = [t for t in order_text if t in set(df["Rating text"])]
counts = df["Rating text"].value_counts().reindex(present)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(counts.index, counts.values, color="teal")
ax.set_title("Restaurant Count by Rating Text Band")
ax.set_xlabel("Rating text")
ax.set_ylabel("Number of restaurants")
fig.tight_layout()
fig.savefig(FIG_DIR / "rating_text_counts.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\1022839740.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Visualization Group 2 — Cuisine and City Average Ratings

Cuisine is the more informative grouping: 145 labels vs 141 cities, but cities are extremely concentrated in New Delhi / Gurgaon / Noida. Both views are shown with documented filters (cuisine `n >= 30`, top 15; cities with `n >= 50`).


In [6]:
cuisine_long = df.dropna(subset=["Cuisines"]).copy()
cuisine_long["Cuisine"] = cuisine_long["Cuisines"].str.split(",")
cuisine_long = cuisine_long.explode("Cuisine")
cuisine_long["Cuisine"] = cuisine_long["Cuisine"].str.strip()
cuisine_long = cuisine_long[cuisine_long["Cuisine"] != ""]
cuisine_long = cuisine_long.drop_duplicates(subset=["Restaurant ID", "Cuisine"])

cuisine_rating = (
    cuisine_long.groupby("Cuisine", as_index=False)
    .agg(n_restaurants=("Restaurant ID", "nunique"), mean_rating=("Aggregate rating", "mean"), total_votes=("Votes", "sum"))
)
eligible = cuisine_rating[cuisine_rating["n_restaurants"] >= MIN_RESTAURANTS]
top_c = eligible.sort_values("mean_rating", ascending=False).head(TOP_N)
top_c.to_csv(TAB_DIR / "viz_top_cuisine_mean_rating.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
plot_c = top_c.iloc[::-1]
ax.barh(plot_c["Cuisine"], plot_c["mean_rating"], color="darkorange")
ax.set_title(f"Average Aggregate Rating by Cuisine (top {TOP_N}, n ≥ {MIN_RESTAURANTS})")
ax.set_xlabel("Mean Aggregate rating")
ax.set_ylabel("Cuisine")
ax.set_xlim(0, 5)
fig.tight_layout()
fig.savefig(FIG_DIR / "avg_rating_by_cuisine.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\1984302344.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
city_stats = (
    df.groupby("City", as_index=False)
    .agg(n_restaurants=("Restaurant ID", "count"), mean_rating=("Aggregate rating", "mean"))
)
city_eligible = city_stats[city_stats["n_restaurants"] >= 50].sort_values("mean_rating", ascending=False)
city_eligible.to_csv(TAB_DIR / "viz_city_mean_rating_min50.csv", index=False)
print("Cities with n >= 50:", len(city_eligible))
display(city_eligible)

fig, ax = plt.subplots(figsize=(8, 5))
plot_city = city_eligible.iloc[::-1]
ax.barh(plot_city["City"], plot_city["mean_rating"], color="steelblue")
ax.set_title("Average Aggregate Rating by City (n ≥ 50 restaurants)")
ax.set_xlabel("Mean Aggregate rating")
ax.set_ylabel("City")
ax.set_xlim(0, 5)
fig.tight_layout()
fig.savefig(FIG_DIR / "avg_rating_by_city.png", dpi=150, bbox_inches="tight")
plt.show()


Cities with n >= 50: 4


,City,n_restaurants,mean_rating
50,Gurgaon,1118,2.651431
88,New Delhi,5473,2.438845
89,Noida,1080,2.036204
43,Faridabad,251,1.866932


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\2110681096.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Visualization Group 3 — Features vs Target

Selections follow Level 2 findings: votes, price range, table booking, and online delivery are the restaurant traits most worth plotting against `Aggregate rating`.


In [8]:
plot_df = df.copy()
plot_df["votes_log"] = np.log1p(plot_df["Votes"])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df["votes_log"], plot_df["Aggregate rating"], alpha=0.25, s=12, edgecolors="none")
ax.set_title("Votes vs Aggregate Rating")
ax.set_xlabel("log1p(Votes)")
ax.set_ylabel("Aggregate rating")
fig.tight_layout()
fig.savefig(FIG_DIR / "votes_vs_rating.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\1418142224.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
price = (
    df.groupby("Price range", as_index=False)["Aggregate rating"]
    .agg(mean_rating="mean", n="count", median_rating="median")
)
price.to_csv(TAB_DIR / "viz_rating_by_price_range.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="Price range", y="Aggregate rating", ax=ax, color="steelblue")
ax.set_title("Aggregate Rating by Price Range")
ax.set_xlabel("Price range")
ax.set_ylabel("Aggregate rating")
fig.tight_layout()
fig.savefig(FIG_DIR / "rating_by_price_range_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(price["Price range"].astype(str), price["mean_rating"], color="teal")
ax.set_title("Mean Aggregate Rating by Price Range")
ax.set_xlabel("Price range")
ax.set_ylabel("Mean Aggregate rating")
ax.set_ylim(0, 5)
fig.tight_layout()
fig.savefig(FIG_DIR / "mean_rating_by_price_range.png", dpi=150, bbox_inches="tight")
plt.show()
price


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\3165207834.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\3165207834.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Price range,mean_rating,n,median_rating
0,1,1.999887,4444,2.9
1,2,2.941054,3113,3.3
2,3,3.683381,1408,3.8
3,4,3.817918,586,3.9


In [10]:
book = (
    df.groupby("Has_Table_Booking", as_index=False)["Aggregate rating"]
    .agg(mean_rating="mean", n="count")
)
book["label"] = book["Has_Table_Booking"].map({0: "No table booking", 1: "Has table booking"})
book.to_csv(TAB_DIR / "viz_rating_by_table_booking.csv", index=False)

deliv = (
    df.groupby("Has_Online_Delivery", as_index=False)["Aggregate rating"]
    .agg(mean_rating="mean", n="count")
)
deliv["label"] = deliv["Has_Online_Delivery"].map({0: "No online delivery", 1: "Has online delivery"})
deliv.to_csv(TAB_DIR / "viz_rating_by_online_delivery.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axes[0].bar(book["label"], book["mean_rating"], color=["#4c78a8", "#f58518"])
axes[0].set_title("Mean Rating by Table Booking")
axes[0].set_ylabel("Mean Aggregate rating")
axes[0].set_ylim(0, 5)
axes[1].bar(deliv["label"], deliv["mean_rating"], color=["#4c78a8", "#f58518"])
axes[1].set_title("Mean Rating by Online Delivery")
fig.tight_layout()
fig.savefig(FIG_DIR / "rating_by_booking_and_delivery.png", dpi=150, bbox_inches="tight")
plt.show()

display(book)
display(deliv)


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\1002459293.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Has_Table_Booking,mean_rating,n,label
0,0,2.559359,8393,No table booking
1,1,3.441969,1158,Has table booking


,Has_Online_Delivery,mean_rating,n,label
0,0,2.465296,7100,No online delivery
1,1,3.248837,2451,Has online delivery


## Cross-task Integration

If Task 1 and Task 2 have already been executed, their saved tables are plotted here. If a file is missing, the cell records that and continues — this notebook still stands on the engineered dataset alone.


In [11]:
if T1_METRICS.exists():
    metrics = pd.read_csv(T1_METRICS)
    display(metrics)
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(metrics))
    width = 0.35
    ax.bar(x - width / 2, metrics["MAE"], width, label="MAE", color="steelblue")
    ax.bar(x + width / 2, metrics["RMSE"], width, label="RMSE", color="darkorange")
    ax.set_xticks(x)
    ax.set_xticklabels(metrics["Model"], rotation=15)
    ax.set_title("Task 1 Recap: Test MAE and RMSE by Model")
    ax.set_ylabel("Error (rating points)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "task1_model_error_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(metrics["Model"], metrics["R2"], color="teal")
    ax.set_title("Task 1 Recap: Test R² by Model")
    ax.set_ylabel("R²")
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "task1_model_r2_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Task 1 metrics not found at", T1_METRICS)
    print("Run Task-1 notebook first to include the ML recap charts.")


,Model,MAE,MSE,RMSE,R2
0,Linear Regression,1.033287,1.552047,1.245812,0.318114
1,Decision Tree,0.284720,0.190387,0.436334,0.916354
2,Random Forest,0.196822,0.090970,0.301612,0.960033


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\1410450622.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\1410450622.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
if T2_STATS.exists():
    cstats = pd.read_csv(T2_STATS)
    top_votes = cstats.sort_values("total_votes", ascending=False).head(TOP_N).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top_votes["Cuisine"], top_votes["total_votes"], color="steelblue")
    ax.set_title("Task 2 Recap: Top Cuisines by Associated Votes")
    ax.set_xlabel("Total associated votes")
    ax.set_ylabel("Cuisine")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "task2_top_cuisines_votes.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Task 2 cuisine stats not found at", T2_STATS)


C:\Users\adity\AppData\Local\Temp\ipykernel_25128\316366854.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results


In [13]:
print("Figures written to", FIG_DIR.resolve())
for p in sorted(FIG_DIR.glob("*.png")):
    print("-", p.name)
print()
print("Tables written to", TAB_DIR.resolve())
for p in sorted(TAB_DIR.glob("*.csv")):
    print("-", p.name)


Figures written to C:\Users\adity\Documents\Cognifyz\Level-3\Task-3_Data_Visualization\outputs\figures
- avg_rating_by_city.png
- avg_rating_by_cuisine.png
- mean_rating_by_price_range.png
- rating_by_booking_and_delivery.png
- rating_by_price_range_boxplot.png
- rating_distribution_histogram.png
- rating_text_counts.png
- task1_model_error_comparison.png
- task1_model_r2_comparison.png
- task2_top_cuisines_votes.png
- votes_vs_rating.png

Tables written to C:\Users\adity\Documents\Cognifyz\Level-3\Task-3_Data_Visualization\outputs\tables
- viz_city_mean_rating_min50.csv
- viz_rating_by_online_delivery.csv
- viz_rating_by_price_range.csv
- viz_rating_by_table_booking.csv
- viz_top_cuisine_mean_rating.csv


## Findings

1. **Rating distribution**: a large spike at 0.0 (Not rated) sits beside a mound of scores around 3.x. Mean is pulled down by the unrated mass; median is higher.
2. **Cuisine / city**: after requiring 30 restaurants, cuisine mean ratings occupy a narrower band than raw popularity. Cities with enough rows are few; New Delhi volume does not imply it has the highest mean.
3. **Feature–target**: higher price ranges show higher rating distributions (Level 2). Votes relate positively to rating but unrated rows cluster at near-zero votes. Table booking and online delivery show higher mean ratings; this is association, not a service-quality experiment.
4. **Task 1 recap** (if present): compare MAE/RMSE/R² across Linear Regression, Decision Tree, and Random Forest on the same test split.
5. **Task 2 recap** (if present): vote-based popularity is concentrated in a handful of cuisine labels.


## Limitations

- Top-N charts omit rare cuisines by design.
- City comparisons are underpowered outside the NCR cluster.
- Boxplots include the 0.0 unrated convention, which widens spread at low price ranges.
- Charts do not imply causation.


## Conclusion

The visualizations confirm three project-level observations: ratings are not a smooth bell curve because of the unrated spike; cuisine popularity (votes) and cuisine quality (mean rating) are different rankings; and price range plus service flags remain the clearest observed associates of `Aggregate rating` among the engineered features.
